In [6]:
import pandas as pd
import pymssql

conn = pymssql.connect(server='info-mssql-etd', user='ETD', password='ETD', database='AquiAlerts')
cursor = conn.cursor()

In [7]:

# DEPARTEMENT ET COMMUNE 
df = pd.read_csv("pprn_gaspar.csv", sep=";", dtype=str, encoding="utf-8")

code_insee_dep = ['16','17','19','23','24','33','40','47','64','79','86','87']
df_dep = df[df['CODE INSEE DEPARTEMENT'].isin(code_insee_dep)]


codes_dep = df_dep[['CODE INSEE DEPARTEMENT','NOM DEPARTEMENT']].drop_duplicates()
codes_dep = codes_dep.dropna()
codes_depa = codes_dep
codes_dep = codes_dep.values.tolist()

cursor.execute("DELETE FROM T_RISQUE_RIS")
cursor.execute("DELETE FROM T_MESURE_MES")
cursor.execute("DELETE FROM T_COMMUNE_COM")
cursor.execute("DELETE FROM T_DEPARTEMENT_DEP")
conn.commit()

for code_dep, lib_dep in codes_dep:
    cursor.execute(
        "INSERT INTO T_DEPARTEMENT_DEP (DEP_ID, DEP_LIB) VALUES (%s, %s)",
        (code_dep, lib_dep)
    )
conn.commit()

df_com = df_dep[['CODE INSEE COMMUNE','NOM COMMUNE','CODE INSEE DEPARTEMENT']].drop_duplicates(subset=['CODE INSEE COMMUNE'])
df_com = df_com.dropna()
df_comu = df_com
df_com = df_com.values.tolist()

for code_com, lib_com, code_dep in df_com:
    cursor.execute(
        "INSERT INTO T_COMMUNE_COM (COM_ID, COM_LIB, DEP_REF_COM) VALUES (%s, %s, %s)",
        (code_com, lib_com, code_dep)
    )

conn.commit()



In [8]:
# DATE 
import pyodbc
df_dates = pd.read_csv("catnat_gaspar.csv", sep=";", dtype=str, encoding="utf-8")
df_dates = df_dates[df_dates['cod_commune'].str[:2].isin(code_insee_dep)]

df_dates['dat_deb'] = pd.to_datetime(df_dates['dat_deb'], format='%Y-%m-%d', errors='coerce')
df_dates['dat_fin'] = pd.to_datetime(df_dates['dat_fin'], format='%Y-%m-%d', errors='coerce')
df_dates['dat_jou'] = (df_dates['dat_fin'] - df_dates['dat_deb']).dt.days
df_dates.loc[df_dates['dat_deb'] == df_dates['dat_fin'], 'dat_jou'] = 0

dates_uniques = df_dates[['dat_deb', 'dat_jou']].drop_duplicates().dropna().values.tolist()

cursor.execute("DELETE FROM T_DATE_DAT")
conn.commit()

for debut, duree in dates_uniques:
    cursor.execute(
        "INSERT INTO T_DATE_DAT (DAT_DATE_DEB, DAT_DUREE) VALUES (%s, %s)",
        (debut.to_pydatetime(), int(duree))
    )
conn.commit()


In [9]:
#TYPE 
df_types = pd.read_csv("catnat_gaspar.csv", sep=";", dtype=str, encoding="utf-8")
df_types = df_types[df_types['cod_commune'].str[:2].isin(code_insee_dep)]

types_uniques = df_types['lib_risque_jo'].dropna().drop_duplicates().tolist()

cursor.execute("DELETE FROM T_TYPE_TYP")
conn.commit()

for lib_type in types_uniques:
    cursor.execute("INSERT INTO T_TYPE_TYP (TYP_LIB) VALUES (%s)", (lib_type,))
conn.commit()

In [10]:
#MESURE
df_mesure = pd.read_csv("catnat_gaspar.csv", sep=";", dtype=str, encoding="utf-8")
df_mesure = df_mesure[['cod_commune', 'lib_commune', 'lib_risque_jo', 'dat_deb', 'dat_fin']]
df_mesure = df_mesure[df_mesure['cod_commune'].str[:2].isin(code_insee_dep)]

df_mesure['dat_deb'] = pd.to_datetime(df_mesure['dat_deb'], format='%Y-%m-%d', errors='coerce')
df_mesure['dat_fin'] = pd.to_datetime(df_mesure['dat_fin'], format='%Y-%m-%d', errors='coerce')
df_mesure['dat_jou'] = (df_mesure['dat_fin'] - df_mesure['dat_deb']).dt.days
df_mesure.loc[df_mesure['dat_deb'] == df_mesure['dat_fin'], 'dat_jou'] = 0
df_mesure = df_mesure.dropna(subset=['dat_deb', 'lib_risque_jo'])

cursor.execute("SELECT COM_ID FROM T_COMMUNE_COM")
com_ids = {row[0] for row in cursor.fetchall()}

cursor.execute("SELECT TYP_ID, TYP_LIB FROM T_TYPE_TYP")
typ_map = {lib.strip(): tid for tid, lib in cursor.fetchall()}

cursor.execute("SELECT DAT_ID, DAT_DATE_DEB, DAT_DUREE FROM T_DATE_DAT")
dat_map = {(deb, duree): did for did, deb, duree in cursor.fetchall()}

df_mesure = df_mesure[df_mesure['cod_commune'].astype(int).isin(com_ids)]

cursor.execute("DELETE FROM T_MESURE_MES")
conn.commit()

for _, row in df_mesure.iterrows():
    com_ref = int(row['cod_commune'])
    typ_ref = typ_map.get(row['lib_risque_jo'].strip())
    dat_ref = dat_map.get((row['dat_deb'].to_pydatetime(), int(row['dat_jou'])))

    cursor.execute(
        "INSERT INTO T_MESURE_MES (COM_REF_MES, TYP_REF_MES, DAT_REF_MES) VALUES (%s, %s, %s)",
        (com_ref, typ_ref, dat_ref)
    )
conn.commit()

In [11]:
# RISQUE
df_risque = pd.read_csv("pprn_gaspar.csv", sep=";", dtype=str, encoding="utf-8")
df_risque = df_risque[['CODE INSEE COMMUNE', 'LIBELLE RISQUE 2', 'PRESCRIPTION', 'APPROBATION']]
df_risque = df_risque[df_risque['CODE INSEE COMMUNE'].str[:2].isin(code_insee_dep)]
df_risque = df_risque.dropna(subset=['CODE INSEE COMMUNE', 'LIBELLE RISQUE 2', 'PRESCRIPTION'])

df_risque['PRESCRIPTION'] = pd.to_datetime(df_risque['PRESCRIPTION'], format='%Y-%m-%d', errors='coerce')
df_risque['APPROBATION'] = pd.to_datetime(df_risque['APPROBATION'], format='%Y-%m-%d', errors='coerce')
df_risque['dat_jou'] = (df_risque['APPROBATION'] - df_risque['PRESCRIPTION']).dt.days
df_risque.loc[df_risque['dat_jou'].isna() | (df_risque['dat_jou'] < 0), 'dat_jou'] = 0
df_risque = df_risque.dropna(subset=['PRESCRIPTION'])


cursor.execute("SELECT TYP_ID, TYP_LIB FROM T_TYPE_TYP")
typ_map = {lib.strip(): tid for tid, lib in cursor.fetchall()}

nouveaux_types = set(df_risque['LIBELLE RISQUE 2'].str.strip()) - set(typ_map.keys())
for lib_type in nouveaux_types:
    cursor.execute("INSERT INTO T_TYPE_TYP (TYP_LIB) VALUES (%s)", (lib_type,))
conn.commit()

cursor.execute("SELECT TYP_ID, TYP_LIB FROM T_TYPE_TYP")
typ_map = {lib.strip(): tid for tid, lib in cursor.fetchall()}


cursor.execute("SELECT DAT_ID, DAT_DATE_DEB, DAT_DUREE FROM T_DATE_DAT")
dat_map = {(deb, duree): did for did, deb, duree in cursor.fetchall()}

for debut, duree in df_risque[['PRESCRIPTION', 'dat_jou']].drop_duplicates().values.tolist():
    cle = (debut.to_pydatetime(), int(duree))
    if cle not in dat_map:
        cursor.execute(
            "INSERT INTO T_DATE_DAT (DAT_DATE_DEB, DAT_DUREE) VALUES (%s, %s)",
            cle
        )
conn.commit()

cursor.execute("SELECT DAT_ID, DAT_DATE_DEB, DAT_DUREE FROM T_DATE_DAT")
dat_map = {(deb, duree): did for did, deb, duree in cursor.fetchall()}



cursor.execute("SELECT COM_ID FROM T_COMMUNE_COM")
com_ids = {row[0] for row in cursor.fetchall()}
df_risque = df_risque[df_risque['CODE INSEE COMMUNE'].astype(int).isin(com_ids)]

cursor.execute("DELETE FROM T_RISQUE_RIS")
conn.commit()

for _, row in df_risque.iterrows():
    com_ref = int(row['CODE INSEE COMMUNE'])
    typ_ref = typ_map.get(row['LIBELLE RISQUE 2'].strip())
    dat_ref = dat_map.get((row['PRESCRIPTION'].to_pydatetime(), int(row['dat_jou'])))

    cursor.execute(
        "INSERT INTO T_RISQUE_RIS (COM_REF_RIS, TYP_REF_RIS, DAT_REF_RIS) VALUES (%s, %s, %s)",
        (com_ref, typ_ref, dat_ref)
    )
conn.commit()

In [ ]:
# DONNEES METEOROLOGIQUES
from pathlib import Path
import os
from datetime import datetime

types_meteo = ['Pluie (RR)', 'Température Min (TN)', 'Température Max (TX)', 'Vent Max (FXI)']
typ_map = {}
for t in types_meteo:
    cursor.execute("SELECT TYP_ID FROM T_TYPE_TYP WHERE TYP_LIB = %s", (t,))
    row = cursor.fetchone()
    if row:
        typ_map[t] = int(row[0])
    else:
        cursor.execute("INSERT INTO T_TYPE_TYP (TYP_LIB) VALUES (%s)", (t,))
        conn.commit()
        cursor.execute("SELECT @@IDENTITY")
        typ_map[t] = int(cursor.fetchone()[0])

cursor.execute("SELECT COM_ID FROM T_COMMUNE_COM")
com_ids_valides = {int(row[0]) for row in cursor.fetchall()}

cursor.execute("SELECT DAT_ID, DAT_DATE_DEB FROM T_DATE_DAT WHERE DAT_DUREE = 1")
date_map = {row[1].strftime('%Y%m%dd'): int(row[0]) for row in cursor.fetchall()}

dossier_csv = Path(r"\\iut.bx\Etudiants\Home\cjenipan\Desktop\venvvisu\SAE\donnees_meteo")
code_insee_dep = ['16', '17', '19', '23', '24', '33', '40', '47', '64', '79', '86', '87']

fichiers_meteo = []
if dossier_csv.exists():
    for fichier in dossier_csv.iterdir():
        nom = fichier.name
        if nom.startswith("Q_") and any(f"Q_{dep}_" in nom for dep in code_insee_dep) and "RR-T-Vent" in nom:
            if "latest-2025-2026" in nom or "previous-1950-2024" in nom:
                fichiers_meteo.append(fichier)

for fichier in fichiers_meteo:
    df = pd.read_csv(fichier, sep=";", dtype=str, encoding="utf-8", compression='infer')
    df.columns = df.columns.str.strip()

    df['COM_REF_MES'] = df['NUM_POSTE'].str.slice(0, 5).astype(int)

    df = df[df['COM_REF_MES'].isin(com_ids_valides)]

    if df.empty:
        continue

    dates_fichier = df['AAAAMMJJ'].drop_duplicates()
    for d_str in dates_fichier:
        if d_str not in date_map:
            try:
                dt = datetime.strptime(d_str, '%Y%m%d')
                cursor.execute("INSERT INTO T_DATE_DAT (DAT_DATE_DEB, DAT_DUREE) VALUES (%s, 1)", (dt,))
                conn.commit()
                cursor.execute("SELECT @@IDENTITY")
                date_map[d_str] = int(cursor.fetchone()[0])
            except Exception:
                continue

    df['DAT_REF_MES'] = df['AAAAMMJJ'].map(date_map)
    df = df.dropna(subset=['DAT_REF_MES'])
    df['DAT_REF_MES'] = df['DAT_REF_MES'].astype(int)

    df_melted = pd.melt(
        df,
        id_vars=['COM_REF_MES', 'DAT_REF_MES'],
        value_vars=['RR', 'TN', 'TX', 'FXI'],
        var_name='TYP_LIB_COURT',
        value_name='MES_VALEUR'
    )

    df_melted = df_melted.dropna(subset=['MES_VALEUR'])
    df_melted = df_melted[df_melted['MES_VALEUR'].str.strip() != ""]
    df_melted['MES_VALEUR'] = pd.to_numeric(df_melted['MES_VALEUR'], errors='coerce')
    df_melted = df_melted.dropna(subset=['MES_VALEUR'])

    if df_melted.empty:
        continue

    name_mapping = {
        'RR': 'Pluie (RR)', 'TN': 'Température Min (TN)',
        'TX': 'Température Max (TX)', 'FXI': 'Vent Max (FXI)'
    }
    df_melted['TYP_REF_MES'] = df_melted['TYP_LIB_COURT'].map(name_mapping).map(typ_map)

    tuples_insertion = list(df_melted[['MES_VALEUR', 'COM_REF_MES', 'TYP_REF_MES', 'DAT_REF_MES']].itertuples(index=False, name=None))

    total_lignes = len(tuples_insertion)
    if total_lignes > 0:
        taille_paquet = 5000
        query = "INSERT INTO T_MESURE_MES (MES_VALEUR, COM_REF_MES, TYP_REF_MES, DAT_REF_MES) VALUES (%s, %s, %s, %s)"

        for i in range(0, total_lignes, taille_paquet):
            paquet = tuples_insertion[i:i+taille_paquet]
            cursor.executemany(query, paquet)
            conn.commit()

In [ ]:
# NETTOYAGE
import time

try:
    conn.rollback()
except:
    pass

requete_limiter_paquet = """
WITH CTE_Meteo AS (
    SELECT 
        M.ID_MES,
        ROW_NUMBER() OVER (PARTITION BY C.DEP_REF_COM ORDER BY D.DAT_DATE_DEB DESC) AS Rang
    FROM T_MESURE_MES M
    JOIN T_COMMUNE_COM C ON M.COM_REF_MES = C.COM_ID
    JOIN T_DATE_DAT D    ON M.DAT_REF_MES = D.DAT_ID
)
DELETE TOP (5000) FROM T_MESURE_MES
WHERE ID_MES IN (
    SELECT ID_MES 
    FROM CTE_Meteo 
    WHERE Rang > 100000
);
"""

total_supprime = 0
while True:
    try:
        cursor.execute(requete_limiter_paquet)
        conn.commit()

        lignes_paquet = cursor.rowcount
        total_supprime += lignes_paquet

        if lignes_paquet > 0:
            time.sleep(0.05)
        else:
            break

    except Exception as e:
        print(f"Erreur ({str(e)[:100]})... Réessai dans 1 seconde.")
        try:
            conn.rollback()
        except:
            pass
        time.sleep(1)